In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

OUT = "Output.csv"
SEED = 42

# ---------------- Load ----------------
df = pd.read_csv("Final_Dataset_AIBD.csv", low_memory=False)
sample = df["Unnamed: 0"]
gene_cols = list(df.columns[5:])
X_all = df[gene_cols].copy()
X_all["Age"] = df["Age"]
X_all["Gender_male"] = (df["Gender"] == "male").astype(int)
X_all["Gender_female"] = (df["Gender"] == "female").astype(int)
feat_names = np.array(X_all.columns)
group = np.where(sample.str.startswith("NC"), "Non-COVID (NC)", "COVID (C)")

# Normalisation: signed log1p (expression is heavily skewed, some values are negative)
# then z-score. In the models the scaler sits inside the CV pipeline (no leakage).
slog = lambda a: np.sign(a) * np.log1p(np.abs(a))
def make_norm():
    return [("slog", FunctionTransformer(slog)), ("scale", StandardScaler())]

# Low-expression filter (unsupervised, no labels used): keep genes that are non-zero
# in >= 20% of the samples being analysed. Age / Gender are always kept.
MIN_FRAC = 0.20
is_cov = np.isin(feat_names, ["Age", "Gender_male", "Gender_female"])
def keep_mask(M):
    return is_cov | ((M > 0).mean(axis=0) >= MIN_FRAC)

# ---------------- PCA on whole data (185 samples) ----------------
k_pca = keep_mask(X_all.values)
print(f"PCA features kept: {k_pca.sum()} of {len(k_pca)}")
Z = Pipeline(make_norm()).fit_transform(X_all.values[:, k_pca])
pca = PCA(n_components=10, random_state=SEED).fit(Z)
pcs = pca.transform(Z)
evr = pca.explained_variance_ratio_
pd.DataFrame(pcs, columns=[f"PC{i+1}" for i in range(10)]).assign(
    Sample=sample, Group=group, ICU=df["ICU"]).to_csv(OUT + "pca_scores.csv", index=False)

icu = df["ICU"].fillna("not recorded (NC)")
fig, ax = plt.subplots(1, 3, figsize=(18, 5.2))
cols = {"Non-COVID (NC)": "#9aa0a6", "COVID (C)": "#1f77b4"}
for g, c in cols.items():
    m = group == g
    ax[0].scatter(pcs[m, 0], pcs[m, 1], c=c, s=28, alpha=.8, label=g)
ax[0].set_title("PCA - all 185 samples by group")
cm = {"no": "#2ca02c", "yes": "#d62728", "not recorded (NC)": "#c8c8c8"}
for k, c in cm.items():
    m = (icu == k).values
    ax[1].scatter(pcs[m, 0], pcs[m, 1], c=c, s=28, alpha=.8, label=f"ICU: {k}")
ax[1].set_title("PCA - coloured by ICU status")
for a in ax[:2]:
    a.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)"); a.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)"); a.legend(fontsize=8)
ax[2].bar(range(1, 11), evr * 100, color="#4c78a8", label="per PC")
ax[2].plot(range(1, 11), np.cumsum(evr) * 100, "o-", c="#e45756", label="cumulative")
ax[2].set_xlabel("Principal component"); ax[2].set_ylabel("Variance explained (%)")
ax[2].set_title("Scree plot"); ax[2].legend()
plt.tight_layout(); plt.savefig(OUT + "pca_plot.png", dpi=150); plt.close()

# ---------------- ICU prediction (labelled COVID samples only) ----------------
lab = df["ICU"].notna().values
k_mod = keep_mask(X_all.values[lab])
print(f"Model features kept: {k_mod.sum()} of {len(k_mod)}")
feat_names = feat_names[k_mod]
X = X_all.values[lab][:, k_mod]
y = (df["ICU"][lab] == "yes").astype(int).values
print("Modelling set:", X.shape, "ICU yes =", y.sum(), "no =", (1 - y).sum())

models = {
    "Logistic Regression": lambda: LogisticRegression(C=0.01, max_iter=5000, random_state=SEED),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED),
    "SVM (linear)": lambda: SVC(kernel="linear", C=0.01, random_state=SEED),
}
def score(m, Xt):
    return m.predict_proba(Xt)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xt)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
grid = np.linspace(0, 1, 101)
res = {}
for name, mk in models.items():
    aucs, accs, tprs = [], [], []
    for tr, te in cv.split(X, y):
        pipe = Pipeline(make_norm() + [("clf", mk())]).fit(X[tr], y[tr])
        s = score(pipe, X[te])
        aucs.append(roc_auc_score(y[te], s))
        accs.append(accuracy_score(y[te], pipe.predict(X[te])))
        f, t, _ = roc_curve(y[te], s)
        tp = np.interp(grid, f, t); tp[0] = 0; tprs.append(tp)
    res[name] = dict(auc=np.array(aucs), acc=np.array(accs), tpr=np.mean(tprs, 0), tpr_sd=np.std(tprs, 0))
    print(f"{name}: AUC {np.mean(aucs):.3f} +/- {np.std(aucs):.3f} | Acc {np.mean(accs):.3f}")

fig, ax = plt.subplots(figsize=(7, 6.2))
for (name, r), c in zip(res.items(), ["#1f77b4", "#2ca02c", "#d62728"]):
    ax.plot(grid, r["tpr"], c=c, lw=2, label=f"{name} (AUC = {r['auc'].mean():.2f} ± {r['auc'].std():.2f})")
    ax.fill_between(grid, np.clip(r["tpr"] - r["tpr_sd"], 0, 1), np.clip(r["tpr"] + r["tpr_sd"], 0, 1), color=c, alpha=.1)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ICU admission - ROC (5-fold CV x 3 repeats, mean ± SD)")
ax.legend(loc="lower right", fontsize=9); plt.tight_layout(); plt.savefig(OUT + "roc_curves.png", dpi=150); plt.close()

pd.DataFrame([{"Model": n, "Mean AUC": r["auc"].mean(), "AUC SD": r["auc"].std(),
               "Mean Accuracy": r["acc"].mean()} for n, r in res.items()]).to_csv(OUT + "model_performance.csv", index=False)

# ---------------- Feature importance (fit on all 100 labelled samples) ----------------
imp = {}
for name, mk in models.items():
    pipe = Pipeline(make_norm() + [("clf", mk())]).fit(X, y)
    clf = pipe.named_steps["clf"]
    if name == "Random Forest":
        imp[name] = clf.feature_importances_
    else:
        imp[name] = clf.coef_.ravel()          # sign: + => higher in ICU
full = pd.DataFrame({"Feature": feat_names})
for n, v in imp.items():
    full[n] = v
    full[n + " rank"] = pd.Series(np.abs(v)).rank(ascending=False).astype(int).values
full.to_csv(OUT + "feature_importance_all.csv", index=False)

top = 20
fig, ax = plt.subplots(1, 3, figsize=(18, 7))
for a, (n, v), c in zip(ax, imp.items(), ["#1f77b4", "#2ca02c", "#d62728"]):
    idx = np.argsort(np.abs(v))[::-1][:top][::-1]
    if n == "Random Forest":
        a.barh(feat_names[idx], v[idx], color=c)
        a.set_xlabel("Gini importance")
    else:
        a.barh(feat_names[idx], v[idx], color=np.where(v[idx] > 0, "#d62728", "#1f77b4"))
        a.set_xlabel("Coefficient (red = higher in ICU, blue = lower)")
    a.set_title(f"{n} - top {top} features")
plt.tight_layout(); plt.savefig(OUT + "feature_importance.png", dpi=150); plt.close()

tops = {n: set(feat_names[np.argsort(np.abs(v))[::-1][:50]]) for n, v in imp.items()}
common = set.intersection(*tops.values())
print("Top-50 overlap across all 3 models:", sorted(common))
for n, v in imp.items():
    print(n, list(feat_names[np.argsort(np.abs(v))[::-1][:10]]))
print("PCA var:", np.round(evr[:5] * 100, 2), "cum10:", round(evr.sum() * 100, 1))

PCA features kept: 16819 of 17724
Model features kept: 15723 of 17724
Modelling set: (100, 15723) ICU yes = 50 no = 50
Logistic Regression: AUC 0.921 +/- 0.044 | Acc 0.853
Random Forest: AUC 0.890 +/- 0.062 | Acc 0.797
SVM (linear): AUC 0.925 +/- 0.039 | Acc 0.857
Top-50 overlap across all 3 models: ['TNIP3']
Logistic Regression ['GNG4', 'COL1A2', 'TMEM191B', 'ARNT2', 'PNMA6A', 'RAB3A', 'TNIP3', 'ZSCAN5B', 'CYP27B1', 'GLDN']
Random Forest ['IL18BP', 'TNIP3', 'CX3CR1', 'PARP3', 'ME3', 'NOD1', 'FCER1A', 'IL4I1', 'SEPTIN8', 'UBA7']
SVM (linear) ['GNG4', 'COL1A2', 'ARNT2', 'PNMA6A', 'TMEM191B', 'TAL2', 'CASP14', 'RAB3A', 'PLCXD3', 'CYP27B1']
PCA var: [29.93 16.4   3.7   2.91  2.39] cum10: 62.6


In [3]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import (confusion_matrix, roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score)

SEED = 42

# ---------- Load: keep only samples with an ICU label ----------
df = pd.read_csv("Final_Dataset_AIBD.csv", low_memory=False)
df = df[df.ICU.notna()].reset_index(drop=True)

X = df.iloc[:, 5:].copy()                       # gene columns
X["Age"] = df["Age"]
X["Gender_male"] = (df.Gender == "male").astype(int)
X["Gender_female"] = (df.Gender == "female").astype(int)
y = (df.ICU == "yes").astype(int).values

# Drop low-expression genes (non-zero in < 20% of samples); no labels used
keep = ((X > 0).mean() >= 0.20) | X.columns.isin(["Age", "Gender_male", "Gender_female"])
X = X.loc[:, keep].values

# ---------- Normalisation: signed log1p, then z-score (inside the pipeline) ----------
slog = lambda a: np.sign(a) * np.log1p(np.abs(a))
def pipe(clf):
    return Pipeline([("slog", FunctionTransformer(slog)),
                     ("scale", StandardScaler()),
                     ("clf", clf)])

models = {
    "Logistic Regression (all genes)": lambda: LogisticRegression(C=0.01, max_iter=5000, random_state=SEED),
    "SVM (linear) (all genes)":        lambda: SVC(kernel="linear", C=0.01, random_state=SEED),
    "Random Forest (all genes)":       lambda: RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=SEED),
}

def score(m, Xt):
    return m.predict_proba(Xt)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(Xt)

def metrics_row(name, y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall (Sensitivity)": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "F1 score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

# ---------- 1) Hold-out test set (n = 37) ----------
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.37, stratify=y, random_state=SEED)
holdout = []
for name, mk in models.items():
    p = pipe(mk()).fit(Xtr, ytr)
    holdout.append(metrics_row(name, yte, p.predict(Xte), score(p, Xte)))
H = pd.DataFrame(holdout)

# ---------- 2) 5-fold CV x 3 repeats (all 100 samples) ----------
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_rows = []
for name, mk in models.items():
    ys, ps, ss, aucs = [], [], [], []
    for tr, te in cv.split(X, y):
        p = pipe(mk()).fit(X[tr], y[tr])
        s = score(p, X[te])
        ys += list(y[te]); ps += list(p.predict(X[te])); ss += list(s)
        aucs.append(roc_auc_score(y[te], s))
    r = metrics_row(name, np.array(ys), np.array(ps), np.array(ss))
    r["ROC-AUC"] = np.mean(aucs)                  # mean of per-fold AUCs
    for k in ["TN", "FP", "FN", "TP"]:
        r[k] = round(r[k] / 3)                    # counts per full pass over 100 samples
    cv_rows.append(r)
C = pd.DataFrame(cv_rows)

pd.set_option("display.width", 250, "display.max_columns", 30)
print("Hold-out test set (n=%d)" % len(yte)); print(H.round(4).to_string(index=False))
print("\n5-fold CV x3"); print(C.round(4).to_string(index=False))
H.round(4).to_csv("holdout_metrics.csv", index=False)
C.round(4).to_csv("cv_metrics.csv", index=False)

Hold-out test set (n=37)
                          Model  Accuracy  Precision  Recall (Sensitivity)  Specificity  F1 score  ROC-AUC  TN  FP  FN  TP
Logistic Regression (all genes)    0.8649        1.0                0.7222       1.0000    0.8387   0.9357  19   0   5  13
       SVM (linear) (all genes)    0.8649        1.0                0.7222       1.0000    0.8387   0.9620  19   0   5  13
      Random Forest (all genes)    0.7568        0.8                0.6667       0.8421    0.7273   0.8713  16   3   6  12

5-fold CV x3
                          Model  Accuracy  Precision  Recall (Sensitivity)  Specificity  F1 score  ROC-AUC  TN  FP  FN  TP
Logistic Regression (all genes)    0.8533     0.8841                0.8133       0.8933    0.8472   0.9207  45   5   9  41
       SVM (linear) (all genes)    0.8567     0.8794                0.8267       0.8867    0.8522   0.9253  44   6   9  41
      Random Forest (all genes)    0.7967     0.7871                0.8133       0.7800    0.8000   